# Single-example / all-examples Dynamic NIAH analysis

This notebook analyzes one selected example from a Dynamic NIAH v2 JSONL dataset. By default, it uses the configured example run under `data/niah-example/{DATASET_RUN_NAME}/dynamic_niah_v2.jsonl`; set `DATASET_RUN_NAME = None` to fall back to `data/niah-example/dynamic_niah_v2.jsonl`. Set `ALL_EXAMPLES = True` to run the same pipeline for every row in the selected JSONL. In all-examples mode, the run folder contains one `example_id_*` subfolder per row plus an aggregate `ablation_results_all.csv`. Runtime artifacts are written under `/content/{RUN_NAME}` and the final zip is copied to `RESULTS_PATH`.


## 1. Mount Drive and choose the repo

Set `REPO_DIR` to your checked-out repository. If you cloned into `/content`, point `REPO_DIR` there instead of Drive.


In [1]:
from google.colab import drive
from pathlib import Path
import os
import sys

drive.mount('/content/drive')

# CHANGE THIS to your checked-out dataset-generation repository.
REPO_DIR = Path('/content/drive/MyDrive/Colab Notebooks/compression/dataset-generation-main-v9')
if not REPO_DIR.exists():
    raise FileNotFoundError(
        f'REPO_DIR does not exist: {REPO_DIR}'
        'Update REPO_DIR to your dataset-generation checkout before continuing.'
    )

os.chdir(REPO_DIR)
if str(REPO_DIR / 'src') not in sys.path:
    sys.path.insert(0, str(REPO_DIR / 'src'))

print('Working directory:', Path.cwd())


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Working directory: /content/drive/MyDrive/Colab Notebooks/compression/dataset-generation-main-v9


## 2. Install dependencies


In [2]:
!pip -q install -U transformers accelerate datasets sentencepiece tqdm matplotlib


## 3. Settings

Set `DATASET_RUN_NAME` to the folder under `data/niah-example` that contains the Dynamic NIAH v2 dataset you want to analyze, or set it to `None` to use the root example dataset. Change `EXAMPLE_ID` to analyze a different zero-based row when `ALL_EXAMPLES = False`. Set `ALL_EXAMPLES = True` to loop over every row in `DATASET_PATH`; each row writes its generated outputs under `/content/{RUN_NAME}/example_id_*`. Ablation defaults are read from `configs/ablation.json`, including the first/last-token exclusion window for critical-token selection.


In [3]:
from pathlib import Path

from single_example import (
    DEFAULT_REPRESENTATION_ABLATION_CONFIG_PATH,
    DEFAULT_REPRESENTATION_RESTORE_CONFIG_PATH,
    run_single_example_representation_ablation,
    run_single_example_representation_restore,
)
from single_example.ablation_analysis import (
    DEFAULT_ABLATION_CONFIG_PATH,
    run_single_example_ablation,
    summarize_ablation_results_all,
)
from single_example.single_example_analysis import (
    DEFAULT_DATASET_PATH,
    cleanup_large_tensor_artifacts,
    compute_single_example_hidden_states,
    create_single_example_run,
    load_jsonl_example,
    load_model_and_tokenizer,
    locate_uncontrolled_needle_segments,
    prepare_single_example_dataset,
    render_and_tokenize_messages,
    resolve_niah_example_dataset_path,
    run_single_example_qk_outlier_analysis,
    save_input_metadata,
    zip_single_example_results,
)

MODEL_NAME = 'Qwen/Qwen3-8B'
EXAMPLE_ID = 0
ALL_EXAMPLES = False ### KEY VARIABLE IN WORKFLOW, if true, notebook will loop over examples, time consuming ###

# Set this to a folder under data/niah-example, or None for the root example dataset.
DATASET_RUN_NAME = 'Qwen_Qwen3-8B_task-argmax_prompt-easier_len-1000_needles-3'
RESTORE_DATASET_RUN_NAME = DATASET_RUN_NAME
DATASET_PATH = resolve_niah_example_dataset_path(DATASET_RUN_NAME)

# Use None for an auto-generated timestamped run name.
USER_RUN_NAME = None

# Hidden/QK layers to analyze. Keep this sparse for a lighter sanity check.
ANALYSIS_LAYERS = [4, 8, 12, 16, 20, 24, 28]
HIDDEN_LAYERS = ANALYSIS_LAYERS
QK_LAYERS = ANALYSIS_LAYERS

# Final archive destination. If None, this is set after RUN_NAME is known.
RESULTS_PATH = None

# Token-level ablation settings. Defaults come from configs/ablation.json; set these to override in Colab.
RUN_ABLATION = False  ### KEY VARIABLE IN WORKFLOW ###
ABLATION_CONFIG_PATH = DEFAULT_ABLATION_CONFIG_PATH
NUM_CRITICAL_TOKENS = None  # e.g. 10 to override configs/ablation.json
ABLATION_RANDOM_SEED = None  # e.g. 12345 to override configs/ablation.json
CRITICAL_TOKEN_CALC_LAYER = None  # e.g. 24 to override configs/ablation.json

# Representation-level ablation settings. Defaults come from configs/ablation-representation.json.
RUN_REPRESENTATION_ABLATION = True  ### KEY VARIABLE IN WORKFLOW ###
ABLATION_REPRESENTATION_CONFIG_PATH = DEFAULT_REPRESENTATION_ABLATION_CONFIG_PATH
REPRESENTATION_NUM_CRITICAL_TOKENS = None  # None keeps configs/ablation-representation.json
RANDOMIZE_FROM_TOP_LAYER = None  # None keeps configs/ablation-representation.json

# Representation-level restore settings. Defaults come from configs/ablation-representation-restore.json.
# This corrupts all needle tokens, then restores targeted clean hidden states.
RUN_REPRESENTATION_RESTORE = False
ABLATION_REPRESENTATION_RESTORE_CONFIG_PATH = DEFAULT_REPRESENTATION_RESTORE_CONFIG_PATH
RESTORE_NUM_CRITICAL_TOKENS = None  # None keeps configs/ablation-representation-restore.json
RESTORE_RANDOMIZE_FROM_TOP_LAYER = None  # None keeps configs/ablation-representation-restore.json

# Keep Hugging Face model downloads on Drive so reconnects do not redownload everything.
HF_CACHE_DIR = Path('/content/drive/MyDrive/Colab Notebooks/huggingface_models')
HF_CACHE_DIR.mkdir(parents=True, exist_ok=True)
os.environ['HF_HOME'] = str(HF_CACHE_DIR)
os.environ['TRANSFORMERS_CACHE'] = str(HF_CACHE_DIR)

row, all_rows = load_jsonl_example(DATASET_PATH, EXAMPLE_ID)
paths = create_single_example_run(
    row=row,
    example_id=EXAMPLE_ID,
    model_name=MODEL_NAME,
    run_root='/content',
    user_run_name=USER_RUN_NAME,
)
if RESULTS_PATH is None:
    RESULTS_PATH = Path('results/single-example') / paths.run_name

print(f'Loaded example {EXAMPLE_ID} of {len(all_rows)} from {DATASET_PATH}')
print('DATASET_RUN_NAME:', DATASET_RUN_NAME)
print('RESTORE_DATASET_RUN_NAME:', RESTORE_DATASET_RUN_NAME)
print('ALL_EXAMPLES:', ALL_EXAMPLES)
print('Run name:', paths.run_name)
print('Run dir:', paths.run_dir)
print('Final archive destination:', RESULTS_PATH)


Loaded example 0 of 20 from data/niah-example/Qwen_Qwen3-8B_task-argmax_prompt-easier_len-1000_needles-3/dynamic_niah_v2.jsonl
DATASET_RUN_NAME: Qwen_Qwen3-8B_task-argmax_prompt-easier_len-1000_needles-3
ALL_EXAMPLES: False
Run name: run_20260607_200006_Qwen_Qwen3-8B_task-argmax_example-0_prompt-easier_len-1000_needles-3
Run dir: /content/run_20260607_200006_Qwen_Qwen3-8B_task-argmax_example-0_prompt-easier_len-1000_needles-3
Final archive destination: results/single-example/run_20260607_200006_Qwen_Qwen3-8B_task-argmax_example-0_prompt-easier_len-1000_needles-3


## 4. Define the reusable per-example pipeline

The function below reuses the same single-example workflow for either the selected row or each row in an all-examples loop.


In [4]:
import torch


def run_pipeline_for_example(*, row, example_id, example_paths):
    print(f'\n===== Example {example_id}: {row.get("id")} =====')
    dataset_copy, cfg = prepare_single_example_dataset(
        row=row,
        example_id=example_id,
        dataset_path=DATASET_PATH,
        paths=example_paths,
        model_name=MODEL_NAME,
    )
    print('Saved one-row dataset:', dataset_copy)
    print('Saved run metadata:', example_paths.metadata_path)

    print('CUDA available:', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('GPU:', torch.cuda.get_device_name(0))

    model, tokenizer = load_model_and_tokenizer(cfg, model_name=MODEL_NAME)
    uncontrolled = render_and_tokenize_messages(
        tokenizer, row['uncontrolled_messages'], thinking_mode=cfg.thinking_mode
    )
    controlled = render_and_tokenize_messages(
        tokenizer, row['messages'], thinking_mode=cfg.thinking_mode
    )
    print('Uncontrolled input shape:', tuple(uncontrolled.input_ids.shape))
    print('Controlled input shape:', tuple(controlled.input_ids.shape))

    needle_segments = locate_uncontrolled_needle_segments(
        row=row,
        uncontrolled_input_ids=uncontrolled.input_ids,
        prompt_text=uncontrolled.prompt_text,
        token_offsets=uncontrolled.token_offsets,
        expected_num_needles=len(row.get('needles', [])) or cfg.num_needles,
    )
    print('Needle segments in uncontrolled model input:')
    for segment in needle_segments:
        print(
            f"  {segment.get('needle_id')}: start={segment['start']} "
            f"end={segment['end']} length={segment['length']} "
            f"is_control={segment.get('is_control')}"
        )

    input_metadata_path = save_input_metadata(
        path=example_paths.generate_data_dir / f'inputs_{example_id}.json',
        example_id=example_id,
        row=row,
        model_name=MODEL_NAME,
        uncontrolled_input_ids=uncontrolled.input_ids,
        controlled_input_ids=controlled.input_ids,
        needle_segments=needle_segments,
    )
    print('Saved input metadata:', input_metadata_path)

    hidden_outputs = compute_single_example_hidden_states(
        model=model,
        uncontrolled_input_ids=uncontrolled.input_ids,
        controlled_input_ids=controlled.input_ids,
        row=row,
        paths=example_paths,
        example_id=example_id,
        layers=HIDDEN_LAYERS,
        needle_segments=needle_segments,
    )
    for name, path in hidden_outputs.items():
        print(f'{name}: {path}')

    # Release this model before the Q/K capture stage reloads with its own attention implementation.
    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    qk_summary = run_single_example_qk_outlier_analysis(
        paths=example_paths,
        layers=QK_LAYERS,
        example_id=example_id,
        model_name=MODEL_NAME,
        repo_root=REPO_DIR,
        force_capture=False,
        capture_attn_implementation='sdpa',
    )
    print('Q/K summary:', qk_summary)

    if RUN_ABLATION:
        model, tokenizer = load_model_and_tokenizer(cfg, model_name=MODEL_NAME)
        ablation_summary = run_single_example_ablation(
            paths=example_paths,
            row=row,
            example_id=example_id,
            model=model,
            tokenizer=tokenizer,
            uncontrolled_input_ids=uncontrolled.input_ids,
            needle_segments=needle_segments,
            config_path=ABLATION_CONFIG_PATH,
            num_critical_tokens=NUM_CRITICAL_TOKENS,
            ablation_random_seed=ABLATION_RANDOM_SEED,
            critical_token_calc_layer=CRITICAL_TOKEN_CALC_LAYER,
            dynamic_cfg=cfg,
        )
        print('Ablation summary:', ablation_summary)
        del model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    else:
        ablation_summary = None
        print('Skipping ablation because RUN_ABLATION=False')

    if RUN_REPRESENTATION_ABLATION:
        model, tokenizer = load_model_and_tokenizer(cfg, model_name=MODEL_NAME)
        representation_ablation_summary = run_single_example_representation_ablation(
            paths=example_paths,
            row=row,
            dataset_path=DATASET_PATH,
            example_id=example_id,
            model=model,
            tokenizer=tokenizer,
            uncontrolled_input_ids=uncontrolled.input_ids,
            needle_segments=needle_segments,
            config_path=ABLATION_REPRESENTATION_CONFIG_PATH,
            num_critical_tokens=REPRESENTATION_NUM_CRITICAL_TOKENS,
            randomize_from_top_layer=RANDOMIZE_FROM_TOP_LAYER,
            dynamic_cfg=cfg,
        )
        print('Representation ablation summary:', representation_ablation_summary)
        del model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    else:
        representation_ablation_summary = None
        print('Skipping representation ablation because RUN_REPRESENTATION_ABLATION=False')

    if RUN_REPRESENTATION_RESTORE:
        model, tokenizer = load_model_and_tokenizer(cfg, model_name=MODEL_NAME)
        representation_restore_summary = run_single_example_representation_restore(
            paths=example_paths,
            row=row,
            dataset_path=DATASET_PATH,
            restore_dataset_run_name=RESTORE_DATASET_RUN_NAME,
            example_id=example_id,
            model=model,
            tokenizer=tokenizer,
            uncontrolled_input_ids=uncontrolled.input_ids,
            needle_segments=needle_segments,
            config_path=ABLATION_REPRESENTATION_RESTORE_CONFIG_PATH,
            num_critical_tokens=RESTORE_NUM_CRITICAL_TOKENS,
            randomize_from_top_layer=RESTORE_RANDOMIZE_FROM_TOP_LAYER,
            dynamic_cfg=cfg,
        )
        print('Representation restore summary:', representation_restore_summary)
        del model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    else:
        representation_restore_summary = None
        print('Skipping representation restore because RUN_REPRESENTATION_RESTORE=False')

    removed_paths = cleanup_large_tensor_artifacts(
        example_paths,
        example_id=example_id,
        remove_qk_cache=True,
        remove_hidden_states=True,
    )
    print('Removed large intermediate paths:')
    for path in removed_paths:
        print(' ', path)

    return {
        'example_id': example_id,
        'row_id': row.get('id'),
        'paths': example_paths,
        'qk_summary': qk_summary,
        'ablation_summary': ablation_summary,
        'representation_ablation_summary': representation_ablation_summary,
        'representation_restore_summary': representation_restore_summary,
    }


## 5. Run the analysis

When `ALL_EXAMPLES = False`, this runs the selected `EXAMPLE_ID` directly in the run folder. When `ALL_EXAMPLES = True`, it loops over every row and saves each example under a subfolder named `example_id_*`.


In [5]:
example_summaries = []

if ALL_EXAMPLES:
    for this_example_id, this_row in enumerate(all_rows):
        example_paths = create_single_example_run(
            row=this_row,
            example_id=this_example_id,
            model_name=MODEL_NAME,
            run_root=paths.run_dir,
            user_run_name=f'example_id_{this_example_id}',
        )
        example_summaries.append(
            run_pipeline_for_example(
                row=this_row,
                example_id=this_example_id,
                example_paths=example_paths,
            )
        )
    if RUN_ABLATION:
        summary_path = summarize_ablation_results_all(
            run_dir=paths.run_dir,
            example_dirs=[summary['paths'].run_dir for summary in example_summaries],
        )
        print('Saved all-examples ablation summary:', summary_path)
else:
    example_summaries.append(
        run_pipeline_for_example(row=row, example_id=EXAMPLE_ID, example_paths=paths)
    )

print('Completed examples:', [summary['example_id'] for summary in example_summaries])



===== Example 0: dynamic_niah_v2_1 =====
Saved one-row dataset: /content/run_20260607_200006_Qwen_Qwen3-8B_task-argmax_example-0_prompt-easier_len-1000_needles-3/generate_data/dynamic_niah_v2.jsonl
Saved run metadata: /content/run_20260607_200006_Qwen_Qwen3-8B_task-argmax_example-0_prompt-easier_len-1000_needles-3/run_metadata.json
CUDA available: True
GPU: NVIDIA A100-SXM4-80GB


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

Uncontrolled input shape: (1, 1140)
Controlled input shape: (1, 1140)
Needle segments in uncontrolled model input:
  N1: start=152 end=183 length=31 is_control=True
  N2: start=283 end=302 length=19 is_control=False
  N3: start=502 end=531 length=29 is_control=False
Saved input metadata: /content/run_20260607_200006_Qwen_Qwen3-8B_task-argmax_example-0_prompt-easier_len-1000_needles-3/generate_data/inputs_0.json
[hidden-analysis] token-length mismatch/alignment  normal_len=1140 control_len=1140 insertion_position=152 offset=0
measurements: /content/run_20260607_200006_Qwen_Qwen3-8B_task-argmax_example-0_prompt-easier_len-1000_needles-3/tensors/inputs_0.pt
hidden: /content/run_20260607_200006_Qwen_Qwen3-8B_task-argmax_example-0_prompt-easier_len-1000_needles-3/tensors/hidden_inputs_0.pt
figure: /content/run_20260607_200006_Qwen_Qwen3-8B_task-argmax_example-0_prompt-easier_len-1000_needles-3/figures/inputs_0.png
input_ids_table: /content/run_20260607_200006_Qwen_Qwen3-8B_task-argmax_examp

Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

[hook] saved layer=4 q_raw shape=(1, 1140, 4096) dtype=torch.bfloat16 -> /content/run_20260607_200006_Qwen_Qwen3-8B_task-argmax_example-0_prompt-easier_len-1000_needles-3/tensors/qk_cache/input_0/layer_04_q_raw.pt
[hook] saved layer=4 k_raw shape=(1, 1140, 1024) dtype=torch.bfloat16 -> /content/run_20260607_200006_Qwen_Qwen3-8B_task-argmax_example-0_prompt-easier_len-1000_needles-3/tensors/qk_cache/input_0/layer_04_k_raw.pt
[hook] saved layer=8 q_raw shape=(1, 1140, 4096) dtype=torch.bfloat16 -> /content/run_20260607_200006_Qwen_Qwen3-8B_task-argmax_example-0_prompt-easier_len-1000_needles-3/tensors/qk_cache/input_0/layer_08_q_raw.pt
[hook] saved layer=8 k_raw shape=(1, 1140, 1024) dtype=torch.bfloat16 -> /content/run_20260607_200006_Qwen_Qwen3-8B_task-argmax_example-0_prompt-easier_len-1000_needles-3/tensors/qk_cache/input_0/layer_08_k_raw.pt
[hook] saved layer=12 q_raw shape=(1, 1140, 4096) dtype=torch.bfloat16 -> /content/run_20260607_200006_Qwen_Qwen3-8B_task-argmax_example-0_promp

Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

Representation ablation summary: {'config': {'num_critical_tokens': 10, 'randomize_from_top_layer': True, 'ablation_random_seed': 12345, 'critical_token_calc_layer': 24, 'patterns': ('massive_activation', 'attention_sink', 'needle_sensitive', 'needle_span', 'massive_activation_all', 'attention_sink_all', 'needle_sensitive_all'), 'attention_sink_score': 'received_uniform_ratio', 'max_new_tokens': None, 'temperature': 0.0, 'stats_dtype': 'bfloat16', 'stats_accum_dtype': 'float32', 'stats_split': 'latter_half', 'profile_allow_single_example': False, 'min_std': 0.0, 'edge_exclusion_tokens': 5, 'save_unablated_hidden_states': True}, 'example_id': 0, 'row_id': 'dynamic_niah_v2_1', 'baseline': {'example_id': 0, 'row_id': 'dynamic_niah_v2_1', 'pattern': 'baseline', 'layer_idx': -1, 'randomize_from_top_layer': True, 'num_positions': 0, 'ablated_positions': '[]', 'model_output_text': '{"city":"New York","score":92}', 'parse_mode': 'json', 'exact_match': True, 'accuracy': 1.0, 'seed': 12345}, 'nu

## 6. Zip and export results


In [6]:
archive_path = zip_single_example_results(paths=paths, results_path=RESULTS_PATH)
print('Archive written to:', archive_path)


Archive written to: results/single-example/run_20260607_200006_Qwen_Qwen3-8B_task-argmax_example-0_prompt-easier_len-1000_needles-3/run_20260607_200006_Qwen_Qwen3-8B_task-argmax_example-0_prompt-easier_len-1000_needles-3.zip


## 7. Optional: release the Colab runtime


In [7]:
# Uncomment to release the Colab VM when the run is complete.
from google.colab import runtime
runtime.unassign()


## Representation-level ablation and restore analysis

Set `RUN_REPRESENTATION_ABLATION = True` in the settings cell to run representation-level corruption ablation inside the per-example pipeline. This mode loads `configs/ablation-representation.json`, profiles hidden-state statistics on the latter half of `DATASET_PATH`, and runs layer-by-layer representation ablations without `model.generate()`.

Set `RUN_REPRESENTATION_RESTORE = True` to run the complementary restore analysis. Restore mode loads `configs/ablation-representation-restore.json`, replaces all needle tokens with irrelevant haystack tokens for each pattern/layer setting, then restores targeted clean hidden states from the uncorrupted prompt. `RESTORE_DATASET_RUN_NAME` is defined right below `DATASET_RUN_NAME` and should point to a folder under `data/niah-example` containing `dynamic_niah_v2.jsonl`.

Before zipping, cleanup removes known intermediate representation tensors and any generated `.pt` file larger than 200 MB.
